In [1]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
import torch

assert torch.cuda.is_available(), "LỖI: Chưa bật T4 GPU! Vào Runtime -> Change runtime type -> T4 GPU."
print(f"GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

zip_file = Path("e2e_eval_v1_v2_colab.zip")
assert zip_file.exists() and zipfile.is_zipfile(zip_file), "LỖI: Chưa upload e2e_eval_v1_v2_colab.zip!"

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "chromadb>=1.0.0", "rank-bm25", "pyyaml",
        "langchain-community", "transformers", "accelerate", "sentence-transformers",
    ],
    check=True,
)

for folder in (
    "Query Understanding and Routing",
    "Retrieval",
    "Response Synthesis",
    "Document Ingestion and Parsing",
    "Evaluation",
):
    subprocess.run(["rm", "-rf", folder], check=True)
Path("path.py").unlink(missing_ok=True)
Path("e2e_v1_vs_v2_all_reports.zip").unlink(missing_ok=True)

subprocess.run(["unzip", "-q", "-o", str(zip_file)], check=True)

root = Path.cwd()
eval_dir = root / "Evaluation" / "End-to-end RAG Evals"
v2_snap = eval_dir / "v2_snapshot"

VERSIONED_FILES = [
    "Query Understanding and Routing/query_understanding_routing/ontology.yaml",
    "Query Understanding and Routing/query_understanding_routing/ontology_loader.py",
    "Query Understanding and Routing/query_understanding_routing/routing_module.py",
    "Query Understanding and Routing/query_understanding_routing/normalization_module.py",
    "Query Understanding and Routing/query_understanding_routing/understanding_module.py",
    "Response Synthesis/verifier.py",
]

for rel in VERSIONED_FILES:
    dst = v2_snap / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(root / rel, dst)

def run_isolated_eval(snap_dir: Path, dataset_name: str, out_subdir: str, run_tag: str) -> None:
    for rel in VERSIONED_FILES:
        shutil.copy2(snap_dir / rel, root / rel)

    extra_paths = [
        str(eval_dir),
        str(root / "Response Synthesis"),
        str(root / "Retrieval"),
        str(root / "Query Understanding and Routing"),
        str(root / "Document Ingestion and Parsing" / "Metadata"),
        str(root / "Document Ingestion and Parsing" / "Provenance and Traceability"),
        str(root),
    ]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    prev_pp = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = os.pathsep.join(extra_paths + ([prev_pp] if prev_pp else []))

    proc = subprocess.Popen(
        [
            sys.executable, "-u", str(eval_dir / "eval_e2e.py"),
            "--dataset", str(eval_dir / dataset_name),
            "--out-dir", str(eval_dir / "reports" / out_subdir),
            "--run-tag", run_tag,
        ],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Đợt chạy {run_tag} thoát với lỗi!")

    # Vì chạy bằng subprocess nên khi tiến trình con kết thúc, toàn bộ VRAM GPU tự động giải phóng sạch 100%
    subprocess.run(
        ["zip", "-q", "-r", "e2e_v1_vs_v2_all_reports.zip", "Evaluation/End-to-end RAG Evals/reports"],
        check=True,
    )
    print(f"\n>>> Đã hoàn tất '{run_tag}' và cập nhật e2e_v1_vs_v2_all_reports.zip")

print(">>> Môi trường sẵn sàng! Hãy chạy lần lượt Cell 1 -> Cell 2 -> Cell 3.")

GPU sẵn sàng: Tesla T4
>>> Môi trường sẵn sàng! Hãy chạy lần lượt Cell 1 -> Cell 2 -> Cell 3.


# Chạy 150 mẫu gốc trên phiên bản v2

In [3]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

root = Path.cwd()
eval_dir = root / "Evaluation" / "End-to-end RAG Evals"
eval_script = eval_dir / "eval_e2e.py"
v2_snap = eval_dir / "v2_snapshot"

# 1. Tối ưu VRAM trong eval_e2e.py (dùng chung 1 model bge-m3 + FP16 cho encoder/reranker + giải phóng cache CUDA)
code = eval_script.read_text(encoding="utf-8")

# 1a. Giải phóng bộ nhớ tạm sau mỗi lần Qwen3-4B generate
old_gen_tail = """        self.call_history.append({
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
        })
        return response_text"""

new_gen_tail = """        self.call_history.append({
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
        })
        del inputs, generated_ids, out_ids
        if self.device == "cuda":
            torch.cuda.empty_cache()
        return response_text"""

if old_gen_tail in code and "del inputs, generated_ids, out_ids" not in code:
    code = code.replace(old_gen_tail, new_gen_tail, 1)

# 1b. Dùng chung 1 instance BAAI/bge-m3 (FP16) giữa QueryVectorizer & Verifier/Matcher, và FP16 + batch_size=8 cho Reranker
old_init_block = """    orchestrator = RAGOrchestrator(
        retrieval_system=retrieval_sys,
        synthesizer=ResponseSynthesizer(llm_client=llm_client),
        verifier=EvidenceVerifier(
            similarity_fn=MultilingualSemanticSimilarity(device=device),
            support_threshold=SEMANTIC_SUPPORT_THRESHOLD,
        ),
        traceable_formatter=TraceableFormatter(),
    )
    matcher = SemanticClaimMatcher(device=device)"""

new_init_block = """    if device == "cuda":
        # 1. Ép Reranker chạy FP16 + batch_size=8 để giảm đỉnh VRAM từ 630MB xuống ~110MB
        if hasattr(reranker, "batch_size"):
            reranker.batch_size = 8
        if hasattr(reranker, "_get_model"):
            ce_obj = reranker._get_model()
            if hasattr(ce_obj, "model"):
                ce_obj.model.half()

        # 2. Chia sẻ chung 1 bản BAAI/bge-m3 (FP16) cho Retrieval, Verifier và ClaimMatcher (tiết kiệm ~3.4GB VRAM)
        st_model = vectorizer._get_model() if hasattr(vectorizer, "_get_model") else getattr(vectorizer, "model", None)
        if st_model is not None:
            st_model.half()
            shared_cache = getattr(sys, "_agy_shared_sentence_transformers", None)
            if shared_cache is None:
                shared_cache = {}
                setattr(sys, "_agy_shared_sentence_transformers", shared_cache)
            shared_cache[("BAAI/bge-m3", device)] = st_model
        torch.cuda.empty_cache()

    orchestrator = RAGOrchestrator(
        retrieval_system=retrieval_sys,
        synthesizer=ResponseSynthesizer(llm_client=llm_client),
        verifier=EvidenceVerifier(
            similarity_fn=MultilingualSemanticSimilarity(device=device),
            support_threshold=SEMANTIC_SUPPORT_THRESHOLD,
        ),
        traceable_formatter=TraceableFormatter(),
    )
    matcher = SemanticClaimMatcher(device=device)"""

if old_init_block in code and "shared_cache[(\"BAAI/bge-m3\", device)]" not in code:
    code = code.replace(old_init_block, new_init_block, 1)

eval_script.write_text(code, encoding="utf-8")
print(">>> Đã tối ưu hóa VRAM cho eval_e2e.py thành công!")

# 2. Cập nhật run_isolated_eval bật PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True chống phân mảnh VRAM
VERSIONED_FILES = [
    "Query Understanding and Routing/query_understanding_routing/ontology.yaml",
    "Query Understanding and Routing/query_understanding_routing/ontology_loader.py",
    "Query Understanding and Routing/query_understanding_routing/routing_module.py",
    "Query Understanding and Routing/query_understanding_routing/normalization_module.py",
    "Query Understanding and Routing/query_understanding_routing/understanding_module.py",
    "Response Synthesis/verifier.py",
]

def run_isolated_eval(snap_dir: Path, dataset_name: str, out_subdir: str, run_tag: str) -> None:
    for rel in VERSIONED_FILES:
        shutil.copy2(snap_dir / rel, root / rel)

    extra_paths = [
        str(eval_dir),
        str(root / "Response Synthesis"),
        str(root / "Retrieval"),
        str(root / "Query Understanding and Routing"),
        str(root / "Document Ingestion and Parsing" / "Metadata"),
        str(root / "Document Ingestion and Parsing" / "Provenance and Traceability"),
        str(root),
    ]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    prev_pp = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = os.pathsep.join(extra_paths + ([prev_pp] if prev_pp else []))

    proc = subprocess.Popen(
        [
            sys.executable, "-u", str(eval_script),
            "--dataset", str(eval_dir / dataset_name),
            "--out-dir", str(eval_dir / "reports" / out_subdir),
            "--run-tag", run_tag,
        ],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Đợt chạy {run_tag} thoát với lỗi!")

    subprocess.run(
        ["zip", "-q", "-r", "e2e_v1_vs_v2_all_reports.zip", "Evaluation/End-to-end RAG Evals/reports"],
        check=True,
    )
    print(f"\n>>> Đã hoàn tất '{run_tag}' và cập nhật e2e_v1_vs_v2_all_reports.zip")

# 3. Chạy Đợt 1: 150 mẫu trên phiên bản v2
run_isolated_eval(
    snap_dir=v2_snap,
    dataset_name="e2e_eval_dataset.json",
    out_subdir="v2_regression_150",
    run_tag="v2_regression_150",
)

>>> Đã tối ưu hóa VRAM cho eval_e2e.py thành công!
>>> Bắt đầu End-to-End & System Performance Evaluation (150 mẫu) trên: CUDA
>>> Đang nạp mô hình sinh văn bản 'Qwen/Qwen3-4B' lên cuda...

Loading weights: 100%|██████████| 398/398 [00:31<00:00, 12.74it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 595.78it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 14338.18it/s]
>>> Đang chạy 1 lượt Warm-up mô hình...

>>> [PHA 1] Thực thi toàn trình 150 mẫu và ghi nhận System Performance...
Abstention triggered cho query='Khi sử dụng mật khẩu để xác thực người dùng, mối quan tâm bảo mật chính là gì và một chính sách mật khẩu mạnh giúp đạt được mục tiêu gì?': 0/20 chunks đạt ngưỡng tin cậy tau=0.25.
Task 'task_001' triggered Abstention Policy: 0 chunks met confidence threshold
Requested version(s) ['2048'] not found in catalog
Task 'task_001': requested version not in knowledge base, abstaining
  ... Đã thực thi 10/150 mẫu (133.3s)
Abstention triggered cho query='Trong P

# Chạy 100 mẫu Held-Out mới trên phiên bản v1

In [4]:
run_isolated_eval(
    snap_dir=eval_dir / "v1_snapshot",
    dataset_name="e2e_heldout_100.json",
    out_subdir="v1_heldout_100",
    run_tag="v1_heldout_100",
)

>>> Bắt đầu End-to-End & System Performance Evaluation (100 mẫu) trên: CUDA
>>> Đang nạp mô hình sinh văn bản 'Qwen/Qwen3-4B' lên cuda...

Loading weights: 100%|██████████| 398/398 [00:31<00:00, 12.50it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 798.06it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 12528.44it/s]
>>> Đang chạy 1 lượt Warm-up mô hình...

>>> [PHA 1] Thực thi toàn trình 150 mẫu và ghi nhận System Performance...
Abstention triggered cho query='Khi cần biết nên tìm hướng dẫn triển khai một yêu cầu bảo mật cụ thể hay hướng dẫn kiểm thử yêu cầu đó, nên phân biệt phạm vi của các loại tài liệu hỗ trợ như thế nào?': 0/20 chunks đạt ngưỡng tin cậy tau=0.25.
Task 'task_001' triggered Abstention Policy: 0 chunks met confidence threshold
  ... Đã thực thi 10/100 mẫu (117.5s)
Requested version(s) ['5.0.0'] not found in catalog
Task 'task_001': requested version not in knowledge base, abstaining
Requested version(s) ['5.0.0'] not found in catalog
Task '

# Chạy 100 mẫu Held-Out mới trên phiên bản v2

In [5]:
run_isolated_eval(
    snap_dir=v2_snap,
    dataset_name="e2e_heldout_100.json",
    out_subdir="v2_heldout_100",
    run_tag="v2_heldout_100",
)

from google.colab import files
files.download("e2e_v1_vs_v2_all_reports.zip")

>>> Bắt đầu End-to-End & System Performance Evaluation (100 mẫu) trên: CUDA
>>> Đang nạp mô hình sinh văn bản 'Qwen/Qwen3-4B' lên cuda...

Loading weights: 100%|██████████| 398/398 [00:31<00:00, 12.44it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 979.51it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 16096.79it/s]
>>> Đang chạy 1 lượt Warm-up mô hình...

>>> [PHA 1] Thực thi toàn trình 150 mẫu và ghi nhận System Performance...
Abstention triggered cho query='Khi cần biết nên tìm hướng dẫn triển khai một yêu cầu bảo mật cụ thể hay hướng dẫn kiểm thử yêu cầu đó, nên phân biệt phạm vi của các loại tài liệu hỗ trợ như thế nào?': 0/20 chunks đạt ngưỡng tin cậy tau=0.25.
Task 'task_001' triggered Abstention Policy: 0 chunks met confidence threshold
  ... Đã thực thi 10/100 mẫu (227.3s)
Abstention triggered cho query='According to OWASP ASVS v5.0.0, how should external tools and reports identify a requirement so that references remain tied to the correct standard

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>